# 📘 Notebook 4 — Regressione Logistica: il primo modello di classificazione

Nel notebook precedente abbiamo previsto **numeri** (regressione). Ora prevediamo **categorie**: spam/non spam, malato/sano, gatto/cane. Questa si chiama **classificazione**.

## 🎯 Cosa imparerai qui
1. Differenza tra **regressione** e **classificazione**
2. Perché la regressione lineare **non funziona** per la classificazione
3. La **funzione sigmoide** e la **regressione logistica**
4. Come **valutare** un classificatore: accuracy, precisione, recall, matrice di confusione
5. Esempio concreto: prevedere se un paziente ha una malattia

## 🤔 Quando usare la regressione logistica?
Quando devi rispondere a **domande sì/no**:
- Questa email è spam?
- Questo cliente smetterà di pagare?
- Questo tumore è benigno o maligno?
- Questo studente passerà l'esame?

Nonostante il nome "regressione", è un modello di **classificazione**!

## 1️⃣ Perché la regressione lineare non basta

Proviamo a usare la regressione lineare per un problema di classificazione. **Spoiler: non funziona bene.**

![Classificazione](Media/04/classificazione.png)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Dataset semplice: ore di studio → ha passato l'esame? (0=no, 1=sì)
ore = np.array([1, 2, 2.5, 3, 4, 5, 5.5, 6, 7, 8, 9, 10], dtype=float)
passato = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1])  # 0 o 1

# Proviamo con la regressione lineare
X = ore.reshape(-1, 1)
modello_lineare = LinearRegression().fit(X, passato)

# Plottiamo
x_grid = np.linspace(0, 12, 100).reshape(-1, 1)
y_pred_lin = modello_lineare.predict(x_grid)

plt.figure(figsize=(9, 5))
plt.scatter(ore, passato, c=passato, cmap='RdYlGn', s=100, edgecolor='k')
plt.plot(x_grid, y_pred_lin, 'b-', label='Regressione lineare')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.axhline(0.5, color='red', linestyle='--', label='Soglia 0.5')
plt.xlabel('Ore di studio')
plt.ylabel('Passato (0/1)')
plt.title('Regressione lineare per classificazione → problemi!')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 💡 Problemi:
# 1. La retta esce sopra 1 e sotto 0 → non sono probabilità valide!
# 2. La pendenza non riflette il vero comportamento (passaggio brusco da 0 a 1)
# Serve una funzione che "schiaccia" i valori tra 0 e 1: la SIGMOIDE.

## 2️⃣ La funzione sigmoide

La sigmoide trasforma qualunque numero in un valore tra 0 e 1, perfetto per rappresentare una **probabilità**:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

- per $z$ molto grande positivo → si avvicina a 1
- per $z$ molto grande negativo → si avvicina a 0
- per $z = 0$ → vale esattamente 0.5

![Sigmoide](Media/04/sigmoide.png)


In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)

plt.figure(figsize=(9, 5))
plt.plot(z, sigmoide(z), 'b-', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='soglia 0.5')
plt.axvline(0, color='red', linestyle='--', alpha=0.6)
plt.title('La funzione sigmoide: "schiaccia" i numeri tra 0 e 1')
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Esempi:")
print(f"sigmoide(-5) = {sigmoide(-5):.3f}")
print(f"sigmoide(0)  = {sigmoide(0):.3f}")
print(f"sigmoide(5)  = {sigmoide(5):.3f}")

## 3️⃣ Come funziona la Regressione Logistica

**Idea**: combiniamo regressione lineare e sigmoide.

$$ p = \sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w x + b)}} $$

Il modello calcola $w x + b$ (come la regressione lineare), ma poi passa il risultato attraverso la sigmoide → otteniamo una **probabilità**.

Per decidere la classe finale:
- se $p \geq 0.5$ → classe **1** (positivo)
- se $p < 0.5$  → classe **0** (negativo)

![imput](Media/04/imputX.png)


In [ ]:
from sklearn.linear_model import LogisticRegression

# Riusiamo i dati di prima
X = ore.reshape(-1, 1)
y = passato

# Creiamo e addestriamo il modello
modello_log = LogisticRegression()
modello_log.fit(X, y)

# Calcoliamo le PROBABILITÀ (non solo 0/1)
x_grid = np.linspace(0, 12, 200).reshape(-1, 1)
probabilita = modello_log.predict_proba(x_grid)[:, 1]   # probabilità della classe 1

plt.figure(figsize=(9, 5))
plt.scatter(ore, passato, c=passato, cmap='RdYlGn', s=100, edgecolor='k', label='Studenti')
plt.plot(x_grid, probabilita, 'b-', linewidth=2, label='Probabilità di passare')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Soglia 0.5')
plt.xlabel('Ore di studio')
plt.ylabel('Probabilità di passare')
plt.title('Regressione Logistica: una bella curva a S!')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Esempi di previsione
for o in [2, 4, 5, 7]:
    p = modello_log.predict_proba([[o]])[0, 1]
    classe = modello_log.predict([[o]])[0]
    print(f"{o} ore → probabilità {p:.1%} → previsione: {'PASSATO' if classe == 1 else 'BOCCIATO'}")

## 4️⃣ Esempio realistico: predire una malattia

Usiamo un dataset "reale" famoso: il **Breast Cancer Dataset** integrato in scikit-learn.
Vogliamo prevedere se un tumore è **maligno (0)** o **benigno (1)** a partire da 30 caratteristiche.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Carichiamo il dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print(f"Dimensione: {X.shape[0]} pazienti, {X.shape[1]} caratteristiche")
print(f"Classi: {dict(zip(data.target_names, np.bincount(y)))}")
print(f"\nPrime 5 righe (prime 5 colonne):")
print(X.iloc[:5, :5])

In [ ]:
# Pipeline completa di addestramento

# 1. Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y mantiene la stessa proporzione di classi in train e test
)

# 2. Scaling (importantissimo per la regressione logistica!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 3. Addestriamo il modello
modello = LogisticRegression(max_iter=1000)   # max_iter aumentato per essere sicuri di convergere
modello.fit(X_train_scaled, y_train)

# 4. Previsioni sul test
y_pred = modello.predict(X_test_scaled)
y_proba = modello.predict_proba(X_test_scaled)[:, 1]

print("✅ Modello addestrato!")

## 5️⃣ Come valutare un classificatore: le metriche

Per la classificazione **non basta dire "% di previsioni corrette"**. Serve un'analisi più fine.

### 5.1 Matrice di confusione

Mostra **come** il modello sbaglia:

|  | Predetto Negativo (0) | Predetto Positivo (1) |
|---|---|---|
| **Reale Negativo (0)** | TN (Vero Negativo) ✅ | FP (Falso Positivo) ⚠️ |
| **Reale Positivo (1)** | FN (Falso Negativo) ⚠️ | TP (Vero Positivo) ✅ |

Un FP e un FN **non sono uguali**! In ambito medico:
- **FP**: dico "sano" → spavento e test inutili (meno grave)
- **FN**: dico "non malato" ma è malato → potrebbe morire (gravissimo

![Matrix Confusione](Media/04/Matrixconfusione.png)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
print("Matrice di confusione:")
print(cm)

# Visualizzazione
disp = ConfusionMatrixDisplay(cm, display_labels=['Maligno', 'Benigno'])
disp.plot(cmap='Blues')
plt.title('Matrice di confusione')
plt.show()

### 5.2 Accuracy, Precision, Recall, F1

- **Accuracy** = (TP+TN) / totale → % di previsioni corrette
- **Precision** = TP / (TP+FP) → "quando dico positivo, quante volte ho ragione?"
- **Recall** (Sensibilità) = TP / (TP+FN) → "di tutti i positivi reali, quanti ne ho beccati?"
- **F1** = media armonica di precision e recall

**Quando importa quale?**
- Diagnosi medica (vogliamo trovare TUTTI i malati): puntiamo sul **recall**
- Filtro antispam (non vogliamo bloccare email vere): puntiamo sulla **precision**
- Dataset bilanciato: **F1** o **accuracy**

![PrecisionRecall](Media/04/precisionrecall.png)


![PrecisionRecall](Media/04/precisionrecall.png)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")

print("\n📋 Report completo:")
print(classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno']))

### 5.3 Curva ROC e AUC

Il modello restituisce **probabilità**. La soglia di default è 0.5, ma possiamo cambiarla!

La **curva ROC** mostra le performance per **tutte le soglie possibili**:
- asse X: tasso di falsi positivi
- asse Y: tasso di veri positivi

L'**AUC** (Area Under Curve) riassume tutto: 1 = perfetto, 0.5 = caso casuale.

![ROC](Media/04/ROC.png)


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, soglie = roc_curve(y_test, y_proba)
auc_score = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Modello casuale (AUC = 0.5)')
plt.xlabel('Tasso Falsi Positivi')
plt.ylabel('Tasso Veri Positivi (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 💡 Più la curva è "in alto a sinistra" e più AUC è vicino a 1, meglio è!

## 6️⃣ Cambiare la soglia di decisione

Per default classifichiamo come 1 se `proba >= 0.5`. Ma in casi sensibili (es: medicina) possiamo abbassare la soglia per essere più "prudenti".

![Soglia](Media/04/precision.png)


In [ ]:
# In questo dataset 1 = benigno e 0 = maligno.
# Vogliamo trovare TUTTI i casi maligni (massimo recall sulla classe 0).
# Quindi alziamo la soglia per dichiarare benigno: solo se siamo molto sicuri.

for soglia in [0.3, 0.5, 0.7, 0.9]:
    y_pred_soglia = (y_proba >= soglia).astype(int)
    cm = confusion_matrix(y_test, y_pred_soglia)
    tn, fp, fn, tp = cm.ravel()
    print(f"Soglia {soglia}: "
          f"TN={tn:3d}  FP={fp:3d}  FN={fn:3d}  TP={tp:3d}  | "
          f"recall benigno={tp/(tp+fn):.3f}  precision benigno={tp/(tp+fp) if (tp+fp)>0 else 0:.3f}")

# 🔧 PROVA TU: in un'app medica, quale soglia sceglieresti?

## 7️⃣ Un dataset sbilanciato: la trappola dell'accuracy

**Esempio**: dataset con il 99% di email "non spam" e l'1% di spam.

Un modello stupido che dice **sempre "non spam"** ha **99% di accuracy** ma è **inutile** (non becca mai uno spam)!

Per questo l'accuracy da sola può ingannare. Servono **precision** e **recall**.

![accuracy](Media/04/accuracy.png)


In [ ]:
# Simuliamo un dataset sbilanciato
from sklearn.datasets import make_classification

X_unb, y_unb = make_classification(
    n_samples=1000, n_features=10,
    weights=[0.95, 0.05],   # 95% classe 0, 5% classe 1
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X_unb, y_unb, test_size=0.3, random_state=42)

# Modello "stupido" che dice sempre 0
y_pred_stupido = np.zeros(len(y_test))
print("📦 Modello che dice SEMPRE 0:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred_stupido):.3f}  ← sembra ottimo!")
print(f"   Recall:    {recall_score(y_test, y_pred_stupido, zero_division=0):.3f}  ← inutile, non trova nulla")

# Modello vero
modello = LogisticRegression(max_iter=1000)
modello.fit(X_train, y_train)
y_pred = modello.predict(X_test)
print("\n🤖 Modello reale:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"   Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"   Precision: {precision_score(y_test, y_pred):.3f}")

# 💡 Lezione: in dataset sbilanciati, NON guardare solo l'accuracy!

## 🎓 Riepilogo del Notebook 4

**La Regressione Logistica** è il modello base per la classificazione binaria. Usa la **sigmoide** per trasformare un punteggio lineare in una **probabilità**.

![RL](Media/04/RL.png)


### ✅ Quando usarla
- Classificazione **sì/no** (binaria) o multi-classe semplice
- Quando ti serve **interpretabilità** (sapere quali variabili pesano di più)
- Come **baseline** di tutti i problemi di classificazione
- Quando hai dati **(quasi) linearmente separabili**

### ❌ Quando NON usarla
- Problemi molto **non lineari** → meglio Random Forest o reti neurali
- Immagini / testo (in forma raw) → meglio modelli specifici

### 🔑 Concetti chiave
| Concetto | Significato |
|---|---|
| Sigmoide | trasforma in probabilità (0-1) |
| Soglia | linea di confine per decidere la classe |
| Matrice di confusione | mostra TP, FP, FN, TN |
| Precision | quanto sono affidabili i miei "positivi" |
| Recall | quanti positivi reali becco |
| F1 | bilancia precision e recall |
| ROC / AUC | performance al variare della soglia |

👉 **Prossimo notebook (05)**: gli **alberi decisionali** e la **Random Forest** — modelli che gestiscono benissimo le non-linearità!